# KHỞI TẠO MÔI TRƯỜNG — VS CODE

In [ ]:
# ── Cài thư viện nếu chưa có (chạy 1 lần, sau đó comment lại) ────────────────
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "squarify", "lunardate", "-q"])

# ── Import ─────────────────────────────────────────────────────────────────────
import os, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import squarify
from lunardate import LunarDate
from datetime import timedelta
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ── Style toàn cục ─────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlepad":     10,
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "figure.dpi":        120,
})
sns.set_palette("muted")

COLOR_ACCENT = "#E05A2B"
COLOR_OK     = "#2E86AB"
COLOR_WARN   = "#F4A261"
COLOR_GOOD   = "#52B788"

# ── Đường dẫn dữ liệu (VS Code local) ─────────────────────────────────────────
DATA_PATH = r"C:\Users\PHUONG ANH\Datathon2026_DataFusion4\data\processed"

# ── Giải nén file zip (chỉ chạy lần đầu) ──────────────────────────────────────
for f in ["ops_master.zip", "sales_master.zip", "customer_master.zip"]:
    fpath = os.path.join(DATA_PATH, f)
    if os.path.exists(fpath):
        with zipfile.ZipFile(fpath) as z:
            z.extractall(DATA_PATH)
        print(f"✅ Extracted: {f}")
    else:
        print(f"⚠️  Không thấy {f} — bỏ qua")

print("\n✅ Import & setup hoàn tất!")

# TẢI DỮ LIỆU VÀ XỬ LÝ

In [ ]:
# ── Tải dữ liệu ───────────────────────────────────────────────────────────────
print("Đang tải dữ liệu...")

sales_master = pd.read_csv(os.path.join(DATA_PATH, "sales_master.csv"), parse_dates=["order_date"])
ops_master   = pd.read_csv(os.path.join(DATA_PATH, "ops_master.csv"),   parse_dates=["order_date"])
inventory    = pd.read_csv(os.path.join(DATA_PATH, "inventory.csv"),    parse_dates=["snapshot_date"])
web_traffic  = pd.read_csv(os.path.join(DATA_PATH, "web_traffic.csv"),  parse_dates=["date"])
sales        = pd.read_csv(os.path.join(DATA_PATH, "sales.csv"),        parse_dates=["Date"])

# ── Biến phụ trợ dùng chung ────────────────────────────────────────────────────
ops_master["year"]      = ops_master["order_date"].dt.year
ops_master["month"]     = ops_master["order_date"].dt.month
ops_master["yearmonth"] = ops_master["order_date"].dt.to_period("M")

sales_master["year"]      = sales_master["order_date"].dt.year
sales_master["month"]     = sales_master["order_date"].dt.month
sales_master["yearmonth"] = sales_master["order_date"].dt.to_period("M")

inventory["yearmonth"] = inventory["snapshot_date"].dt.to_period("M")
inventory["month"]     = inventory["snapshot_date"].dt.month

web_traffic["year_month"] = web_traffic["date"].dt.to_period("M")
inventory["year_month"]   = inventory["snapshot_date"].dt.to_period("M")

# ── Merge ops + sales (dùng cho Descriptive) ──────────────────────────────────
ops_sales = ops_master.merge(
    sales_master[[
        "order_id","category","segment","size","price","cogs",
        "quantity","discount_amount"
    ]].drop_duplicates("order_id"),
    on="order_id", how="left"
)
ops_sales["net_revenue"] = (
    ops_sales["quantity"] * ops_sales["price"] - ops_sales["discount_amount"]
).fillna(0)

# ── Xử lý dữ liệu dashboard cũ ────────────────────────────────────────────────
# Tầng 1: Doanh thu Q1 theo mốc 10 ngày
q1_sales = sales[sales["Date"].dt.month.isin([1,2,3])].copy()

def get_10day_period(day):
    if day <= 10:  return "Ngày 1"
    elif day <= 20: return "Ngày 11"
    else:           return "Ngày 21"

q1_sales["Day_Bin"]      = q1_sales["Date"].dt.day.apply(get_10day_period)
q1_sales["Month"]        = q1_sales["Date"].dt.month
month_dict               = {1:"Th1", 2:"Th2", 3:"Th3"}
q1_sales["Period_Label"] = q1_sales["Month"].map(month_dict) + "\n(" + q1_sales["Day_Bin"] + ")"
q1_agg = q1_sales.groupby(["Month","Day_Bin","Period_Label"])["Revenue"].mean().reset_index()
q1_agg = q1_agg.sort_values(by=["Month","Day_Bin"])

# Tầng 2: Traffic vs Stockout
waste_analysis = (
    web_traffic.groupby("year_month")["sessions"].sum().reset_index()
    .merge(inventory.groupby("year_month")["stockout_flag"].sum().reset_index(), on="year_month")
    .tail(15)
)
waste_analysis["month_year"] = waste_analysis["year_month"].dt.strftime("%m/%Y")

# Tầng 3: Lagged Correlation
daily_traffic              = web_traffic.groupby("date")["sessions"].sum().reset_index()
daily_data                 = sales.merge(daily_traffic, left_on="Date", right_on="date", how="inner")
daily_data["sessions_lag_1"] = daily_data["sessions"].shift(1)
corr_matrix_1              = daily_data[["Revenue","sessions","sessions_lag_1"]].corr()
corr_matrix_1.columns      = ["Doanh thu","Truy cập (S)","Truy cập (Hôm qua)"]
corr_matrix_1.index        = ["Doanh thu","Truy cập (S)","Truy cập (Hôm qua)"]

# Tầng 4-6: Dashboard cũ cột phải
product_mix  = sales_master.groupby(["category","segment"])["price"].sum().reset_index()
returns_data = ops_master[ops_master["return_count"] > 0].merge(
    sales_master[["order_id","segment"]].drop_duplicates(), on="order_id")
return_agg   = (returns_data.groupby(["segment","main_return_reason"])["order_id"]
                .count().reset_index().rename(columns={"order_id":"count"}))
device_trend = (sales_master
    .groupby([sales_master["order_date"].dt.to_period("M"), "device_type"])["order_id"]
    .count().unstack().fillna(0).tail(12))
device_trend.index      = device_trend.index.strftime("%m/%Y")
device_trend_pct        = device_trend.div(device_trend.sum(axis=1), axis=0) * 100

print("\n✅ Tải dữ liệu xong!")
print(f"   sales_master : {sales_master.shape}")
print(f"   ops_master   : {ops_master.shape}")
print(f"   inventory    : {inventory.shape}")
print(f"   web_traffic  : {web_traffic.shape}")
print(f"   sales        : {sales.shape}")

# DASHBOARD VẬN HÀNH & KINH DOANH (CŨ)

In [ ]:
# ════════════════════════════════════════════════════════════
# DASHBOARD CŨ — GIỮ NGUYÊN
# ════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(24, 26))
gs  = gridspec.GridSpec(3, 2, figure=fig, wspace=0.25, hspace=0.4)
fig.suptitle("Đánh giá vận hành và kinh doanh", fontsize=28, fontweight="bold", y=0.94)

# CỘT TRÁI
ax1 = fig.add_subplot(gs[0, 0])
sns.lineplot(data=q1_agg, x="Period_Label", y="Revenue",
             color="#c0392b", marker="o", linewidth=3, ax=ax1)
ax1.set_title("Xu hướng doanh thu", fontsize=18, fontweight="bold", color="#2c3e50")
ax1.set_ylabel("Doanh Thu Trung Bình")
ax1.set_xlabel("")
ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))

ax2      = fig.add_subplot(gs[1, 0])
ax2_twin = ax2.twinx()
sns.barplot(data=waste_analysis, x="month_year", y="stockout_flag",
            color="#e74c3c", alpha=0.6, ax=ax2)
sns.lineplot(data=waste_analysis, x="month_year", y="sessions",
             color="#2980b9", marker="s", linewidth=3, ax=ax2_twin)
ax2.set_title("Lãng phí Marketing do đứt gãy tồn kho",
              fontsize=18, fontweight="bold", color="#2c3e50")
ax2.set_ylabel("Số lượng hết Hàng", color="#c0392b")
ax2_twin.set_ylabel("Lượng Truy Cập",  color="#2980b9")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=45)

ax3 = fig.add_subplot(gs[2, 0])
sns.heatmap(corr_matrix_1, annot=True, cmap="Blues", fmt=".2f",
            annot_kws={"size":16}, ax=ax3, cbar=False)
ax3.set_title("Lưu lượng web dẫn dắt Sales", fontsize=18, fontweight="bold", color="#2c3e50")

# CỘT PHẢI
ax4 = fig.add_subplot(gs[0, 1])
squarify.plot(sizes=product_mix["price"], label=product_mix["category"],
              alpha=.9, color=sns.color_palette("YlGnBu", len(product_mix)),
              ax=ax4, text_kwargs={"fontsize":14, "weight":"bold"})
ax4.set_title("Cấu trúc dòng tiền theo cơ cấu sản phẩm",
              fontsize=18, fontweight="bold", color="#2c3e50")
ax4.axis("off")

ax5 = fig.add_subplot(gs[1, 1])
sns.barplot(data=return_agg.sort_values("count", ascending=False).head(10),
            y="segment", x="count", hue="main_return_reason", palette="Set2", ax=ax5)
ax5.set_title("Rò rỉ biên lợi nhuận do Trả hàng",
              fontsize=18, fontweight="bold", color="#2c3e50")
for container in ax5.containers:
    ax5.bar_label(container, padding=5, fmt="%.0f", fontweight="bold")

ax6 = fig.add_subplot(gs[2, 1])
device_trend_pct.plot(kind="area", stacked=True, ax=ax6, alpha=0.8, colormap="Paired")
ax6.set_title("Xu hướng thanh toán sang Mobile",
              fontsize=18, fontweight="bold", color="#2c3e50")
ax6.set_ylabel("Tỷ trọng (%)")
ax6.set_xlabel("Tháng")
ax6.legend(loc="lower right")
ax6.set_xticks(range(len(device_trend_pct.index)))
ax6.set_xticklabels(device_trend_pct.index, rotation=45)

plt.savefig("Dashboard_Vertical_Q1_Solar.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard cũ đã vẽ xong")

***Làm thế nào để đồng bộ hóa nỗ lực kéo khách (Marketing) với khả năng đáp ứng hàng hóa (Kho vận) nhằm tránh lãng phí ngân sách và tối đa hóa dòng tiền trong các sự kiện mua sắm lớn?***

**Phân tích chi tiết từng biểu đồ**

***1. Xu hướng doanh thu Quý 1***

- Biểu đồ đường (Line chart) thể hiện doanh thu trung bình theo các mốc 10 ngày (Thượng - Trung - Hạ tuần). Góc nhìn này cực kỳ quan trọng vì nó làm rõ các điểm rơi của dòng tiền mà dữ liệu theo tháng thông thường bị che khuất.

- Biểu đồ này cho thấy doanh thu luôn tạo đỉnh bùng nổ trước Tết và ngay lập tức chạm đáy đứt gãy trong tuần nghỉ lễ; vì vậy chúng tôi đề nghị chia nhỏ ngân sách tiếp thị theo từng chu kỳ 10 ngày, dồn lực bứt tốc trước Tết và cắt giảm hoàn toàn Ads lúc nghỉ lễ.

***2. Lãng phí Marketing do đứt gãy tồn kho***

- Biểu đồ trục kép kết hợp giữa lượng truy cập (đường) và số ngày hết hàng (cột).

- Biểu đồ này cho thấy chúng ta đang ném tiền qua cửa sổ khi Marketing kéo khách vào nhưng Kho lại không có hàng để bán; vì vậy chúng tôi đề nghị thiết lập hệ thống cảnh báo tồn kho tự động trước khi chạy các chiến dịch Siêu Sale số đôi.

***3. Lưu lượng web dẫn dắt Sales***

- Biểu đồ này cho thấy khách hàng thường mất từ 24h-48h lướt web tham khảo trước khi chốt đơn; vì vậy chúng tôi đề nghị sử dụng dữ liệu truy cập có độ trễ (Lag_1) làm chỉ báo sớm để dự báo chính xác khối lượng đơn hàng ngày mai.

**Đề xuất:** Tự động nâng mức dự trữ an toàn (Safety Stock) lên thêm 30% trước các đợt Siêu Sale số đôi và trước thềm Tết Âm lịch, đồng thời chỉ duyệt giải ngân chi phí chạy quảng cáo khi hệ thống kho báo cáo trữ lượng hàng hóa đã sẵn sàng trên 80%.

***Tại sao dòng sản phẩm bán chạy nhất lại là nguyên nhân bào mòn biên lợi nhuận, và sự dịch chuyển hành vi thanh toán của khách hàng đang mang lại rủi ro nào?***

**1. Cấu trúc dòng tiền theo cơ cấu sản phẩm** — Treemap cho thấy phân khúc Streetwear đang là huyết mạch nuôi sống toàn bộ doanh nghiệp; đề nghị dồn ưu tiên QC cho dòng sản phẩm này.

**2. Rò rỉ biên lợi nhuận do Trả hàng** — Streetwear bán chạy nhất nhưng bị hoàn trả nhiều nhất do "wrong_size"; đề nghị R&D thiết kế lại bảng thông số kích cỡ.

**3. Xu hướng thanh toán sang Mobile** — COD trên mobile mang rủi ro "bùng hàng" cực cao; đề nghị tung mã giảm giá ép khách thanh toán qua thẻ/ví điện tử.

---
# 📊 DESCRIPTIVE ANALYSIS — TẦNG 1/4
## Câu hỏi kinh doanh: "Vòng lặp thất thoát lợi nhuận trong vận hành đang xảy ra ở quy mô nào?"

> **Luồng phân tích:**
> - **D1** → 4 KPI cards: quy mô thiệt hại tổng thể là bao nhiêu?
> - **D2** → Phân bổ trả hàng: loại hàng nào bị trả, vì lý do gì?
> - **D3** → Baseline tồn kho: fill_rate & stockout theo tháng — có lệch pha với traffic không?
>
> *Kết thúc D3 → tín hiệu chuyển sang **Diagnostic**: "Nếu wrong_size chiếm đa số TRẢ HÀNG và fill_rate thấp đúng tháng TRAFFIC CAO → đây không phải ngẫu nhiên."*

## D1 — KPI Tổng quan vận hành

In [ ]:
# ════════════════════════════════════════════════════════════
# D1 — KPI TỔNG QUAN VẬN HÀNH
# Câu hỏi: "Thiệt hại tổng thể từ vận hành kém là bao nhiêu?"
# ════════════════════════════════════════════════════════════

# ── Tính KPI ───────────────────────────────────────────────────────────────────
total_orders       = len(ops_master)
orders_with_return = (ops_master["return_count"] > 0).sum()
return_rate_pct    = orders_with_return / total_orders * 100
total_refund       = ops_master["total_refund"].sum()
total_net_rev      = ops_sales["net_revenue"].sum()
refund_over_rev    = total_refund / total_net_rev * 100
avg_rating         = ops_master["avg_rating"].dropna().mean()
pct_low_rating     = (ops_master["avg_rating"].dropna() < 3).mean() * 100

ops_ship = ops_master.dropna(subset=["ship_date","delivery_date"]).copy()
ops_ship["delivery_days"] = (ops_ship["delivery_date"] - ops_ship["order_date"]).dt.days
late_rate_pct = (ops_ship["delivery_days"] > 7).mean() * 100

print(f"Tổng đơn         : {total_orders:,}")
print(f"Đơn có trả hàng  : {orders_with_return:,} ({return_rate_pct:.1f}%)")
print(f"Tổng refund      : {total_refund:,.0f} VNĐ ({refund_over_rev:.1f}% doanh thu)")
print(f"Avg rating       : {avg_rating:.2f}/5  ({pct_low_rating:.1f}% đơn rating < 3)")
print(f"Tỷ lệ giao trễ   : {late_rate_pct:.1f}%")

# ── Vẽ KPI cards ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle("D1 — Bức tranh tổng thể: Chi phí ẩn từ vận hành kém hiệu quả",
             fontsize=14, fontweight="bold", y=1.02)

kpis = [
    {"ax":axes[0], "value":f"{return_rate_pct:.1f}%",
     "sub":f"{orders_with_return:,} / {total_orders:,} đơn",
     "label":"Tỷ lệ đơn bị trả hàng",
     "color":COLOR_ACCENT, "bench":"Benchmark ngành: ~5%"},
    {"ax":axes[1], "value":f"{refund_over_rev:.1f}%",
     "sub":f"{total_refund/1e9:.1f} tỷ VNĐ hoàn trả",
     "label":"Refund / Doanh thu thuần",
     "color":COLOR_WARN,   "bench":"Mỗi 1% = hàng tỷ VNĐ mất"},
    {"ax":axes[2], "value":f"{avg_rating:.2f} ★",
     "sub":f"{pct_low_rating:.1f}% đơn rating < 3",
     "label":"Rating trung bình",
     "color":COLOR_OK if avg_rating >= 4 else COLOR_WARN,
     "bench":"Target: ≥ 4.0 / 5.0"},
    {"ax":axes[3], "value":f"{late_rate_pct:.1f}%",
     "sub":"Ngưỡng SLA: 7 ngày",
     "label":"Tỷ lệ giao hàng trễ",
     "color":COLOR_ACCENT if late_rate_pct > 20 else COLOR_WARN,
     "bench":"Target: < 15%"},
]

for k in kpis:
    ax = k["ax"]
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    card = mpatches.FancyBboxPatch((0.05,0.05),0.9,0.9,
        boxstyle="round,pad=0.02",
        facecolor=k["color"]+"18", edgecolor=k["color"],
        linewidth=1.8, transform=ax.transAxes)
    ax.add_patch(card)
    ax.text(0.5,0.80,k["value"],ha="center",va="center",
            fontsize=28,fontweight="bold",color=k["color"],transform=ax.transAxes)
    ax.text(0.5,0.56,k["sub"],  ha="center",va="center",
            fontsize=9, color="#555",transform=ax.transAxes)
    ax.text(0.5,0.38,k["label"],ha="center",va="center",
            fontsize=10,fontweight="bold",color="#222",transform=ax.transAxes)
    ax.text(0.5,0.18,k["bench"],ha="center",va="center",
            fontsize=8, style="italic",color="#888",transform=ax.transAxes)

plt.tight_layout()
plt.savefig("D1_kpi_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 INSIGHT D1:
   Refund chiếm {:.1f}% doanh thu, rating trung bình {:.2f}/5,
   {:.1f}% đơn giao trễ → vấn đề lan rộng cả sản phẩm lẫn logistics.
🔍 DẪN VÀO D2: Ta cần biết LOẠI HÀNG nào bị trả nhiều nhất
   và LÝ DO gì chủ đạo.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""".format(refund_over_rev, avg_rating, late_rate_pct))

### D1 — Nhận xét

Bốn chỉ số KPI cho thấy quy mô thiệt hại đang ở mức đáng lo ngại. Tỷ lệ refund chiếm một phần không nhỏ trong doanh thu thuần, đồng thời rating dưới 3 sao và tỷ lệ giao trễ đều vượt ngưỡng an toàn — cho thấy vấn đề không chỉ nằm ở chất lượng sản phẩm mà còn ở logistics.

**→ Câu hỏi tiếp theo:** Loại hàng nào đang là nguồn gốc chính của các khoản refund này?

## D2 — Phân bổ trả hàng theo danh mục × lý do

In [ ]:
# ════════════════════════════════════════════════════════════
# D2 — PHÂN BỔ TRẢ HÀNG THEO DANH MỤC × LÝ DO
# Câu hỏi: "Hàng nào bị trả, vì lý do gì, và mỗi lý do
#           gây tổn thất tài chính bao nhiêu?"
# ════════════════════════════════════════════════════════════

df_ret = ops_sales[ops_sales["return_count"] > 0].copy()

# Return rate theo category
total_by_cat   = ops_sales.groupby("category")["order_id"].count().rename("total")
returns_by_cat = df_ret.groupby("category")["order_id"].count().rename("returned")
cat_rate = pd.concat([total_by_cat, returns_by_cat], axis=1).fillna(0)
cat_rate["return_rate"] = cat_rate["returned"] / cat_rate["total"] * 100
cat_rate = cat_rate.sort_values("return_rate", ascending=False)

# Refund theo category × lý do
ret_reason = (df_ret.groupby(["category","main_return_reason"])
              .agg(count=("order_id","count"), refund=("total_refund","sum"))
              .reset_index())
pivot_refund = ret_reason.pivot_table(
    index="category", columns="main_return_reason", values="refund", fill_value=0)
pivot_refund_pct = pivot_refund.div(pivot_refund.sum(axis=1), axis=0) * 100

top_reasons = (ret_reason.groupby("main_return_reason")["count"]
               .sum().nlargest(5).index.tolist())

# ── Vẽ ─────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle("D2 — Phân bổ trả hàng: Danh mục nào bị trả nhiều & vì lý do gì?",
             fontsize=14, fontweight="bold", y=1.02)

# Panel 1: Return rate theo category
ax1 = fig.add_subplot(gs[0,0])
colors_bar = [COLOR_ACCENT if v > cat_rate["return_rate"].median()
              else COLOR_OK for v in cat_rate["return_rate"]]
bars = ax1.barh(cat_rate.index, cat_rate["return_rate"],
                color=colors_bar, edgecolor="white", linewidth=0.8, height=0.65)
for bar, v in zip(bars, cat_rate["return_rate"]):
    ax1.text(v+0.2, bar.get_y()+bar.get_height()/2,
             f"{v:.1f}%", va="center", fontsize=9, color="#333")
med = cat_rate["return_rate"].median()
ax1.axvline(med, color="#888", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.text(med+0.1, -0.6, f"Median {med:.1f}%", fontsize=8, color="#888", style="italic")
ax1.set_xlabel("Return rate (%)")
ax1.set_title("Return rate theo danh mục", pad=8)
ax1.invert_yaxis()

# Panel 2: Stacked bar số đơn trả theo lý do × category
ax2 = fig.add_subplot(gs[0,1])
plot_data  = ret_reason[ret_reason["main_return_reason"].isin(top_reasons)]
pivot_cnt  = plot_data.pivot_table(
    index="category", columns="main_return_reason", values="count", fill_value=0)
pivot_cnt  = pivot_cnt.reindex(cat_rate.index)
pivot_cnt.plot(kind="barh", ax=ax2, stacked=True,
               color=sns.color_palette("tab10", len(top_reasons)),
               edgecolor="white", linewidth=0.5, width=0.65)
ax2.set_xlabel("Số đơn bị trả")
ax2.set_title("Lý do trả hàng theo danh mục\n(top 5 lý do)", pad=8)
ax2.legend(loc="lower right", fontsize=8, framealpha=0.7)
ax2.invert_yaxis()

# Panel 3: Heatmap % refund
ax3 = fig.add_subplot(gs[0,2])
pivot_show = pivot_refund_pct[
    [c for c in top_reasons if c in pivot_refund_pct.columns]
].reindex(cat_rate.index)
sns.heatmap(pivot_show, ax=ax3, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label":"% refund của category"},
            annot_kws={"size":9})
ax3.set_title("% Tổng refund theo lý do\n(trong từng danh mục)", pad=8)
ax3.set_xlabel("Lý do trả hàng")
ax3.set_ylabel("")
ax3.tick_params(axis="x", rotation=30)
ax3.tick_params(axis="y", rotation=0)

plt.savefig("D2_return_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

top_cat    = cat_rate.index[0]
top_reason = df_ret["main_return_reason"].value_counts().index[0]
top_r_pct  = df_ret["main_return_reason"].value_counts(normalize=True).iloc[0]*100
top_refund = df_ret[df_ret["category"]==top_cat]["total_refund"].sum()

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 INSIGHT D2:
   • Category "{top_cat}" có return rate cao nhất
     → tổng refund: {top_refund:,.0f} VNĐ
   • Lý do phổ biến nhất: "{top_reason}" ({top_r_pct:.1f}% đơn trả)
🔍 DẪN VÀO D3: Ta biết AI TRẢ và VÌ SAO. Tiếp theo cần biết
   tồn kho có đang lệch pha với lúc traffic cao nhất không.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

### D2 — Nhận xét

Ba panel cho thấy bức tranh đầy đủ về trả hàng: panel trái xác định danh mục có return rate cao nhất so với median, panel giữa phân tách lý do theo từng danh mục, panel phải lượng hoá thiệt hại tài chính (% refund).

Điểm quan trọng: nếu `wrong_size` chiếm tỷ trọng lớn → đây là vấn đề **có thể giải quyết được** bằng cải tiến bảng size, không phải vấn đề chất lượng sản phẩm.

**→ Câu hỏi tiếp theo:** Stockout có xảy ra đúng lúc traffic cao nhất không?

## D3 — Baseline tồn kho: Fill rate & Stockout

In [ ]:
# ════════════════════════════════════════════════════════════
# D3 — BASELINE TỒN KHO: FILL_RATE & STOCKOUT
# Câu hỏi: "Tồn kho đang phục vụ tốt đến đâu — và có lệch
#           pha với traffic cao không?"
# ════════════════════════════════════════════════════════════

# ── Tổng hợp theo tháng ────────────────────────────────────────────────────────
inv_monthly = (
    inventory.groupby("yearmonth")
    .agg(avg_fill_rate  =("fill_rate",     "mean"),
         total_stockout =("stockout_flag", "sum"),
         n_products     =("product_id",    "nunique"))
    .reset_index()
)
inv_monthly["yearmonth_dt"] = inv_monthly["yearmonth"].dt.to_timestamp()
inv_monthly["stockout_pct"] = inv_monthly["total_stockout"] / inv_monthly["n_products"] * 100

fillrate_by_month = inventory.groupby("month")["fill_rate"].mean()
stockout_by_month = inventory.groupby("month")["stockout_flag"].mean() * 100

month_labels = ["T1","T2","T3","T4","T5","T6","T7","T8","T9","T10","T11","T12"]
bar_colors = [
    COLOR_ACCENT if m in [1,2] else
    COLOR_WARN   if m in [9,10,11,12] else
    COLOR_OK
    for m in range(1,13)
]

# ── Vẽ ─────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("D3 — Baseline tồn kho: Fill rate & Stockout theo thời gian",
             fontsize=14, fontweight="bold", y=1.01)

# Panel trên trái: Fill rate theo tháng trong năm
ax1 = fig.add_subplot(gs[0,0])
bars = ax1.bar(month_labels, fillrate_by_month.values,
               color=bar_colors, edgecolor="white", linewidth=0.8, width=0.7)
ax1.axhline(0.95, color="red",  linestyle="--", linewidth=1.2, label="Target 95%")
ax1.axhline(fillrate_by_month.mean(), color="#555", linestyle=":",
            linewidth=1, label=f"TB: {fillrate_by_month.mean():.2%}")
ax1.set_ylim(0.8, 1.02)
ax1.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax1.set_title("Fill rate trung bình theo tháng\n(tổng hợp tất cả năm)")
patch_tet  = mpatches.Patch(color=COLOR_ACCENT, label="Tết (T1–T2)")
patch_sale = mpatches.Patch(color=COLOR_WARN,   label="Mega Sale (T9–T12)")
patch_norm = mpatches.Patch(color=COLOR_OK,     label="Tháng thường")
ax1.legend(handles=[patch_tet, patch_sale, patch_norm], fontsize=8)

# Panel trên phải: % sản phẩm stockout theo tháng
ax2 = fig.add_subplot(gs[0,1])
ax2.bar(month_labels, stockout_by_month.values,
        color=bar_colors, edgecolor="white", linewidth=0.8, width=0.7)
ax2.set_ylabel("% sản phẩm bị stockout")
ax2.set_title("% Sản phẩm stockout theo tháng\n(tổng hợp tất cả năm)")
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(decimals=1))

# Panel dưới trái: Timeline fill rate 10 năm
ax3      = fig.add_subplot(gs[1,0])
ax3_twin = ax3.twinx()
ax3.plot(inv_monthly["yearmonth_dt"], inv_monthly["avg_fill_rate"],
         color=COLOR_OK, linewidth=1.8, alpha=0.9, label="Avg fill rate")
ax3.axhline(0.95, color="red", linestyle="--", linewidth=1, alpha=0.7, label="Target 95%")
ax3.fill_between(inv_monthly["yearmonth_dt"],
                 inv_monthly["avg_fill_rate"], 0.95,
                 where=inv_monthly["avg_fill_rate"] < 0.95,
                 alpha=0.25, color=COLOR_ACCENT, label="Dưới target")
ax3_twin.bar(inv_monthly["yearmonth_dt"], inv_monthly["stockout_pct"],
             color=COLOR_ACCENT, alpha=0.25, width=20, label="% SP stockout")
ax3.set_ylim(0.75, 1.05)
ax3.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax3.set_ylabel("Fill rate",     color=COLOR_OK)
ax3_twin.set_ylabel("% Stockout", color=COLOR_ACCENT)
ax3.set_title("Fill rate & Stockout — timeline 2012–2022")
ax3.tick_params(axis="x", rotation=30, labelsize=8)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1+lines2, labels1+labels2, fontsize=8, loc="lower left")

# Panel dưới phải: Heatmap fill rate category × tháng
ax4 = fig.add_subplot(gs[1,1])
fillrate_cat = inventory.groupby(["category","month"])["fill_rate"].mean().unstack()
fillrate_cat.columns = month_labels
sns.heatmap(fillrate_cat, ax=ax4, annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=0.7, vmax=1.0,
            linewidths=0.4, cbar_kws={"label":"Fill rate"},
            annot_kws={"size":8})
ax4.set_title("Fill rate theo danh mục × tháng\n(đỏ = nguy hiểm)")
ax4.set_xlabel("Tháng"); ax4.set_ylabel("")
ax4.tick_params(axis="y", rotation=0)

plt.savefig("D3_inventory_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

worst_month = fillrate_by_month.idxmin()
worst_val   = fillrate_by_month.min()
months_low  = (fillrate_by_month < 0.95).sum()
worst_cat   = fillrate_cat.stack().idxmin()

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 INSIGHT D3:
   • Tháng {worst_month} có fill rate thấp nhất: {worst_val:.2%}
   • {months_low}/12 tháng fill rate dưới target 95%
   • Worst case: {worst_cat[0]} tháng {worst_cat[1]}
     fill rate thấp nhất toàn bộ dataset

🔗 KẾT NỐI 3 CELL DESCRIPTIVE:
   D1 → Quy mô: {return_rate_pct:.1f}% đơn trả, {refund_over_rev:.1f}% doanh thu refund
   D2 → Nguyên nhân mặt hàng: {top_reason} chiếm đa số, tập trung ở {top_cat}
   D3 → Nguyên nhân vận hành: fill rate thấp đúng T1–T2 (Tết) và T9–T12 (Mega Sale)

⚠️  CHUYỂN SANG DIAGNOSTIC:
   "Fill rate thấp đúng lúc traffic cao → doanh thu bị bỏ lỡ có thể đo được.
    Tầng Diagnostic sẽ kiểm chứng bằng số cụ thể."
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

### D3 — Nhận xét & Kết nối sang Diagnostic

Bốn panel xác nhận: fill rate luôn thấp hơn target 95% vào đúng **tháng 1–2 (Tết)** và **tháng 9–12 (Mega Sale)** — tức là hệ thống kho đang thất bại đúng lúc nhu cầu và traffic đạt đỉnh.

Kết hợp 3 cell Descriptive:
- **D1** xác nhận quy mô thiệt hại đang ở mức đáng lo ngại
- **D2** chỉ ra `wrong_size` và danh mục cụ thể gây thiệt hại nhiều nhất
- **D3** cho thấy stockout xảy ra đúng lúc traffic cao → doanh thu bị bỏ lỡ

**→ Tầng Diagnostic** sẽ kiểm chứng: (1) return_rate cao nhất ở size nào × category nào, (2) lượng doanh thu bị mất cụ thể khi stockout đúng lúc traffic đỉnh, (3) mối liên hệ rating thấp → trả hàng nhiều.